# BLIND DRILL — 5/5, type it cold

**Rules:** blank cell → write the whole thing from memory → run → *only then* open the Solution.
No hand-holding in the prompts on purpose. The spec tells you **what**, never **how**.
If you can't start, that's the signal to drill that method — not to peek immediately.

New dataset (e-commerce transactions), new methods you haven't grinded yet:
window/rolling, `melt`, `qcut`, `transform`, cyclical encoding, RandomizedSearch,
calibration, PR curves, permutation importance, SQL CTEs/windows, t-test/chi-square/bootstrap.


## 0 — Setup & generate the dataset (just run this)

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(7)
N = 6000

cust = rng.integers(1000, 1400, N)                       # ~400 repeat customers
channel = rng.choice(["web", "app", "store", "phone"], N, p=[.45, .35, .12, .08])
country = rng.choice([f"C{i}" for i in range(40)], N)    # high cardinality

# order timestamps across 2 years (hour-of-day + day-of-week matter)
start = pd.Timestamp("2023-01-01")
offsets = rng.integers(0, 730 * 24, N)                   # hours
ts = start + pd.to_timedelta(offsets, unit="h")

# skewed order value; app orders slightly smaller; some negatives = refunds
value = np.exp(rng.normal(3.8, 0.7, N)) - (channel == "app") * 5
value[rng.random(N) < 0.04] *= -1                        # refunds
items = rng.integers(1, 12, N)

# discount as text pct, some blanks
disc = rng.choice([0, 5, 10, 15, 20, 25], N, p=[.4, .15, .15, .12, .1, .08])
discount_text = np.array([f"{d}%" if rng.random() > .05 else "" for d in disc], dtype=object)

# target: did the customer churn after this order (imbalanced ~15%)
lin = (-(np.log(np.abs(value) + 1) - 3.8) + (channel == "phone") * 1.2
       + (disc == 0) * 0.5)
p = 1 / (1 + np.exp(-(lin - 1.9)))
churn = (rng.random(N) < p).astype(int)

df = pd.DataFrame({
    "customer_id": cust, "order_ts": ts, "channel": channel, "country": country,
    "order_value": value.round(2), "n_items": items,
    "discount_text": discount_text, "churn": churn,
}).sort_values("order_ts").reset_index(drop=True)
print(df.shape, "| churn rate:", round(df["churn"].mean(), 3))
df.head()

(6000, 8) | churn rate: 0.2


,customer_id,order_ts,channel,country,order_value,n_items,discount_text,churn
0,1252,2023-01-01 01:00:00,app,C38,154.12,11,10%,0
1,1177,2023-01-01 03:00:00,store,C32,172.42,6,0%,0
2,1370,2023-01-01 04:00:00,store,C7,72.68,9,0%,0
3,1168,2023-01-01 05:00:00,app,C0,17.39,10,20%,0
4,1322,2023-01-01 08:00:00,phone,C1,43.99,1,0%,1


In [7]:
# EDA
print(df.nunique().sum())
print(df.describe())
df.info()
print(df.value_counts(["churn"], normalize=True))


10260
       customer_id                    order_ts  order_value      n_items  \
count  6000.000000                        6000  6000.000000  6000.000000   
mean   1200.580667  2024-01-09 03:44:01.800000    51.065213     6.026167   
min    1000.000000         2023-01-01 01:00:00  -201.020000     1.000000   
25%    1101.750000         2023-07-16 20:45:00    24.520000     3.000000   
50%    1199.000000         2024-01-10 16:30:00    41.110000     6.000000   
75%    1301.000000         2024-07-05 18:00:00    69.000000     9.000000   
max    1399.000000         2024-12-30 19:00:00   454.260000    11.000000   
std     114.877996                         NaN    48.470925     3.179229   

             churn  
count  6000.000000  
mean      0.200000  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       0.000000  
max       1.000000  
std       0.400033  
<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column         Non-N

---
# Part 1 — Advanced pandas

### Exercise - Per-customer running order count & cumulative spend

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 8 min
> Skill: Querying & retrieval
> ```

For each row, add `order_seq` (1,2,3… = the customer's Nth order in time order) and `cum_spend` (that customer's cumulative `order_value` up to and including this order). Data is already time-sorted.


In [33]:
# df.groupby("customer_id")[["order_ts", "order_value"]].transform(lambda x: x.sort_values(by=["order_ts"]).cumsum())
df["order_seq"] = df.groupby("customer_id").cumcount()+1
df["order_seqw"] = df.groupby("customer_id")["customer_id"].transform(lambda x: pd.Series(np.arange(len(x))+1, x.index))
df["order_seqw"] = df.groupby("customer_id")["customer_id"].transform(lambda x: np.arange(1,len(x)+1))
df["cum_spend"] = df.groupby("customer_id")["order_value"].cumsum()
df["cum_spendw"] = df.groupby("customer_id")["order_value"].transform(lambda x: x.cumsum())

<details><summary>Solution</summary>

```python
df["order_seq"] = df.groupby("customer_id").cumcount() + 1
df["cum_spend"] = df.groupby("customer_id")["order_value"].cumsum()
df[["customer_id", "order_ts", "order_value", "order_seq", "cum_spend"]].head(10)
```

</details>

### Exercise - Rolling 3-order average value per customer

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 8 min
> Skill: Querying & retrieval
> ```

Add `roll3_value` = the trailing mean of the last 3 `order_value`s **per customer** (min 1 observation). Must not leak future orders.


In [ ]:
df["roll3_value"] = (df.groupby("customer_id")["order_value"]
                     .transform(lambda x:
                                x.rolling(3).mean()))

: 

<details><summary>Solution</summary>

```python
df["roll3_value"] = (
    df.groupby("customer_id")["order_value"]
      .transform(lambda s: s.rolling(3, min_periods=1).mean())
)
df[["customer_id", "order_value", "roll3_value"]].head(10)
```

</details>

### Exercise - Value tier within channel (group-relative qcut)

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 7 min
> Skill: Querying & retrieval
> ```

Create `value_tier` = quartile label (`Q1`..`Q4`) of `order_value`, computed **within each channel** (so each channel has its own quartile cutoffs). Drop refunds (negative value) for the binning; tier them as `NA`.


In [ ]:
def q4(s):
    return pd.qcut(s, 4, labels=["Q1", "Q2", "Q3", "Q4"])


<details><summary>Solution</summary>

```python
def q4(s):
    return pd.qcut(s, 4, labels=["Q1", "Q2", "Q3", "Q4"])
pos = df["order_value"] > 0
df["value_tier"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[pos, "value_tier"] = (
    df.loc[pos].groupby("channel")["order_value"].transform(q4).astype(object)
)
df[["channel", "order_value", "value_tier"]].head(10)
```

</details>

### Exercise - Wide channel-spend matrix, then melt back

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 8 min
> Skill: Querying & retrieval
> ```

Build a wide table: index `customer_id`, columns = `channel`, values = total `order_value` (fill missing 0). Then `melt` it back to long form with columns `customer_id, channel, total_value` and keep only nonzero rows.


<details><summary>Solution</summary>

```python
wide = df.pivot_table(index="customer_id", columns="channel",
                      values="order_value", aggfunc="sum", fill_value=0)
long = wide.reset_index().melt(id_vars="customer_id",
                               var_name="channel", value_name="total_value")
long = long[long["total_value"] != 0].reset_index(drop=True)
long.head()
```

</details>

---
# Part 2 — Feature engineering

### Exercise - Datetime parts + cyclical hour encoding

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 8 min
> Skill: Data cleaning & preprocessing
> ```

From `order_ts` derive: `hour`, `dow` (0–6), `month`, `is_weekend` (0/1). Then encode `hour` cyclically as `hour_sin`, `hour_cos` (so 23:00 is close to 00:00).


<details><summary>Solution</summary>

```python
df["hour"] = df["order_ts"].dt.hour
df["dow"] = df["order_ts"].dt.dayofweek
df["month"] = df["order_ts"].dt.month
df["is_weekend"] = (df["dow"] >= 5).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df[["order_ts", "hour", "dow", "is_weekend", "hour_sin", "hour_cos"]].head()
```

</details>

### Exercise - Parse discount text + interaction feature

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 6 min
> Skill: Data cleaning & preprocessing
> ```

Turn `discount_text` (e.g. `"15%"`, `""`) into numeric `discount` (blank → 0). Then build interaction `value_per_item` = `order_value` / `n_items`.


<details><summary>Solution</summary>

```python
df["discount"] = (
    df["discount_text"].str.replace("%", "", regex=False)
      .replace("", "0").astype(float)
)
df["value_per_item"] = df["order_value"] / df["n_items"]
df[["discount_text", "discount", "order_value", "n_items", "value_per_item"]].head()
```

</details>

### Exercise - Recency/Frequency/Monetary (RFM) per customer

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 10 min
> Skill: Feature engineering
> ```

Build a customer-level RFM table: `recency_days` (days from customer's last order to ref date `2025-01-01`), `frequency` (order count), `monetary` (mean order_value). Return one row per `customer_id`.


<details><summary>Solution</summary>

```python
ref = pd.Timestamp("2025-01-01")
rfm = df.groupby("customer_id").agg(
    last_order=("order_ts", "max"),
    frequency=("order_ts", "size"),
    monetary=("order_value", "mean"),
)
rfm["recency_days"] = (ref - rfm["last_order"]).dt.days
rfm = rfm.drop(columns="last_order").reset_index()
rfm.head()
```

</details>

### Exercise - Polynomial features on two numerics

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 6 min
> Skill: Feature engineering
> ```

Using sklearn, generate degree-2 polynomial features (incl. interaction, no bias) from `order_value` and `n_items`. Show the generated feature names.


<details><summary>Solution</summary>

```python
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
arr = pf.fit_transform(df[["order_value", "n_items"]])
names = pf.get_feature_names_out(["order_value", "n_items"])
pd.DataFrame(arr, columns=names).head()
```

</details>

---
# Part 3 — Modeling depth

Assume a cleaned modeling frame. Build it once, then each exercise stands alone.

<details><summary>Solution — modeling frame setup</summary>

```python
from sklearn.model_selection import train_test_split

feat = df.copy()
feat["discount"] = feat["discount_text"].str.replace("%", "", regex=False).replace("", "0").astype(float)
feat["hour"] = feat["order_ts"].dt.hour
feat["dow"] = feat["order_ts"].dt.dayofweek
num_cols = ["order_value", "n_items", "discount", "hour", "dow"]
cat_cols = ["channel"]
X = feat[num_cols + cat_cols]
y = feat["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X_tr.shape, y_tr.mean().round(3))
```

</details>

### Exercise - RandomizedSearchCV over a pipeline

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 10 min
> Skill: ML & predictive modeling
> ```

Tune a `RandomForestClassifier` inside a Pipeline (numeric scale + one-hot `channel`) with **RandomizedSearchCV** (`n_iter=15`, `cv=5`, scoring `roc_auc`). Search `n_estimators`, `max_depth`, `min_samples_leaf`. Print best params + best score.


<details><summary>Solution</summary>

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe = Pipeline([("pre", pre), ("rf", RandomForestClassifier(random_state=0))])
params = {
    "rf__n_estimators": randint(100, 400),
    "rf__max_depth": randint(3, 20),
    "rf__min_samples_leaf": randint(1, 20),
}
search = RandomizedSearchCV(pipe, params, n_iter=15, cv=5,
                            scoring="roc_auc", random_state=0, n_jobs=-1)
search.fit(X_tr, y_tr)
print(search.best_params_)
print("best AUC:", round(search.best_score_, 4))
```

</details>

### Exercise - Precision-Recall curve + average precision

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 7 min
> Skill: ML & predictive modeling
> ```

Fit any reasonable pipeline. Plot the **precision-recall curve** on the test set and report **average precision** (PR-AUC). State in a comment why PR-AUC beats ROC-AUC here.


<details><summary>Solution</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score
import matplotlib.pyplot as plt

clf = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000,
                class_weight="balanced"))]).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print("avg precision (PR-AUC):", round(average_precision_score(y_te, proba), 4))
PrecisionRecallDisplay.from_predictions(y_te, proba); plt.show()
# at ~15% positives, PR-AUC reflects minority-class performance; ROC-AUC is
# optimistic because true negatives dominate the FPR denominator.
```

</details>

### Exercise - Probability calibration + reliability curve

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 9 min
> Skill: ML & predictive modeling
> ```

Wrap a classifier in `CalibratedClassifierCV` (isotonic, cv=5). Plot the **calibration curve** (reliability diagram) for raw vs calibrated probabilities and compare **Brier scores**.


<details><summary>Solution</summary>

```python
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt

base = Pipeline([("pre", pre), ("rf", RandomForestClassifier(
    n_estimators=200, random_state=0))]).fit(X_tr, y_tr)
cal = CalibratedClassifierCV(base, method="isotonic", cv=5).fit(X_tr, y_tr)

p_raw = base.predict_proba(X_te)[:, 1]
p_cal = cal.predict_proba(X_te)[:, 1]
print("Brier raw :", round(brier_score_loss(y_te, p_raw), 4))
print("Brier cal :", round(brier_score_loss(y_te, p_cal), 4))
fig, ax = plt.subplots()
CalibrationDisplay.from_predictions(y_te, p_raw, n_bins=10, ax=ax, name="raw")
CalibrationDisplay.from_predictions(y_te, p_cal, n_bins=10, ax=ax, name="calibrated")
plt.show()
```

</details>

### Exercise - Permutation importance (model-agnostic)

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 8 min
> Skill: ML & predictive modeling
> ```

Compute **permutation importance** on the test set for a fitted pipeline (scoring `roc_auc`, `n_repeats=10`). Report the top features by mean importance drop. Say why this beats `.feature_importances_`.


<details><summary>Solution</summary>

```python
from sklearn.inspection import permutation_importance

clf = Pipeline([("pre", pre), ("rf", RandomForestClassifier(
    n_estimators=200, random_state=0))]).fit(X_tr, y_tr)
r = permutation_importance(clf, X_te, y_te, scoring="roc_auc",
                           n_repeats=10, random_state=0, n_jobs=-1)
imp = pd.Series(r.importances_mean, index=X_te.columns).sort_values(ascending=False)
print(imp.round(4))
# permutation importance measures impact on the *held-out* metric and is
# model-agnostic; tree .feature_importances_ is train-based and biased toward
# high-cardinality / continuous features.
```

</details>

### Exercise - Soft-voting ensemble vs its members

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 9 min
> Skill: ML & predictive modeling
> ```

Build a soft-voting ensemble of LogisticRegression + RandomForest + GradientBoosting (each in the same `pre`). Compare 5-fold CV ROC-AUC of the ensemble against each member.


<details><summary>Solution</summary>

```python
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score

lr = Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000))])
rf = Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=200, random_state=0))])
gb = Pipeline([("pre", pre), ("m", GradientBoostingClassifier(random_state=0))])
vote = VotingClassifier([("lr", lr), ("rf", rf), ("gb", gb)], voting="soft")

for name, m in [("lr", lr), ("rf", rf), ("gb", gb), ("vote", vote)]:
    s = cross_val_score(m, X_tr, y_tr, cv=5, scoring="roc_auc")
    print(f"{name:5s} AUC {s.mean():.4f} +/- {s.std():.4f}")
```

</details>

---
# Part 4 — SQL + statistics

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
df.to_sql("orders", conn, index=False, if_exists="replace")
def q(sql): return pd.read_sql(sql, conn)
q("SELECT COUNT(*) AS n FROM orders")

### Exercise - CTE + window: each order's running share of customer spend

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 10 min
> Skill: Querying & retrieval (SQL)
> ```

In SQL: for each order, show `customer_id`, `order_value`, and `pct_of_customer` = order_value divided by that customer's total spend, as a %. Use a CTE for the per-customer totals (or a window SUM). Limit 10.


<details><summary>Solution</summary>

```python
q('''
WITH tot AS (
  SELECT customer_id, SUM(order_value) AS total
  FROM orders GROUP BY customer_id
)
SELECT o.customer_id, o.order_value,
       ROUND(100.0 * o.order_value / t.total, 2) AS pct_of_customer
FROM orders o
JOIN tot t ON o.customer_id = t.customer_id
ORDER BY o.customer_id
LIMIT 10
''')
```

</details>

### Exercise - Window: rank orders within channel + running total

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 9 min
> Skill: Querying & retrieval (SQL)
> ```

In SQL: per `channel`, rank orders by `order_value` desc (`rk`), and show a running total of `order_value` ordered by value desc within the channel. Top 12 rows.


<details><summary>Solution</summary>

```python
q('''
SELECT channel, order_value,
       RANK() OVER (PARTITION BY channel ORDER BY order_value DESC) AS rk,
       SUM(order_value) OVER (PARTITION BY channel ORDER BY order_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS run_total
FROM orders
ORDER BY channel, rk
LIMIT 12
''')
```

</details>

### Exercise - Two-sample t-test: order value by churn

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 7 min
> Skill: Statistics
> ```

Test whether mean `order_value` differs between churned and non-churned customers. Use Welch's t-test (unequal variance). Report t-stat, p-value, and a one-line verdict at α=0.05.


<details><summary>Solution</summary>

```python
from scipy import stats
a = df.loc[df["churn"] == 1, "order_value"]
b = df.loc[df["churn"] == 0, "order_value"]
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"t={t:.3f}  p={p:.4g}")
print("significant" if p < 0.05 else "not significant", "at alpha=0.05")
```

</details>

### Exercise - Chi-square: channel vs churn independence

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 7 min
> Skill: Statistics
> ```

Test whether `channel` and `churn` are independent. Build the contingency table and run a chi-square test. Report chi2, p, dof, and the verdict.


<details><summary>Solution</summary>

```python
from scipy import stats
ct = pd.crosstab(df["channel"], df["churn"])
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(f"chi2={chi2:.3f}  p={p:.4g}  dof={dof}")
print("dependent" if p < 0.05 else "independent", "at alpha=0.05")
```

</details>

### Exercise - Bootstrap 95% CI for the churn rate

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 9 min
> Skill: Statistics
> ```

Estimate a 95% confidence interval for the overall churn rate via bootstrap (2000 resamples). Report point estimate and the [2.5%, 97.5%] interval. Seed for determinism.


<details><summary>Solution</summary>

```python
rng2 = np.random.default_rng(0)
vals = df["churn"].to_numpy()
n = len(vals)
boot = np.array([rng2.choice(vals, n, replace=True).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"churn rate = {vals.mean():.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
```

</details>

---
# Part 5 — Method recall checklist (cover the right column, recite the left)

| Need | Reach for |
|------|-----------|
| Nth event per group | `groupby.cumcount()+1` |
| Cumulative per group | `groupby[col].cumsum()` |
| Trailing window per group, no leak | `groupby.transform(lambda s: s.rolling(k, min_periods=1).mean())` |
| Group-relative bins | `groupby.transform(lambda s: pd.qcut(s, 4, labels=...))` |
| Long↔wide | `pivot_table` / `melt(id_vars=...)` |
| Cyclical time | `sin/cos(2*pi*x/period)` |
| Poly/interactions | `PolynomialFeatures(degree=2, include_bias=False)` |
| Random hyperparam search | `RandomizedSearchCV(..., n_iter, scipy.stats distributions)` |
| Minority-class metric | `average_precision_score` / PR curve |
| Trustworthy probabilities | `CalibratedClassifierCV` + Brier + reliability curve |
| Model-agnostic importance | `permutation_importance(scoring=..., n_repeats=...)` |
| Combine models | `VotingClassifier(voting="soft")` |
| Per-row share of group total (SQL) | CTE total + JOIN, or `SUM() OVER (PARTITION BY)` |
| Running total (SQL) | `SUM(x) OVER (PARTITION BY .. ORDER BY .. ROWS UNBOUNDED PRECEDING)` |
| Compare two means | Welch `ttest_ind(..., equal_var=False)` |
| Categorical association | `chi2_contingency(crosstab)` |
| CI without a formula | bootstrap percentile |

If any row's right column didn't come to you instantly, that's tomorrow's drill.
